<a href="https://colab.research.google.com/github/SubhadeepBhadra/fake-news-classifier-LSTM/blob/main/FakeNewsClassifierUsingLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fake News Classifier Using LSTM

Dataset: https://www.kaggle.com/c/fake-news/data#

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

In [4]:
df=pd.read_csv('/content/drive/MyDrive/train.csv')

In [5]:
df.head()

,id,title,author,text,label
0,0,Congress Approves Annual Defense Budget,Daniel Martinez,A new study published in the peer-reviewed jou...,0
1,1,Senate Passes Bipartisan Infrastructure Package,Paul Anderson,The annual report released Thursday demonstrat...,0
2,2,Study Finds Mediterranean Diet Linked to Longe...,Steven Robinson,"In a press conference held Monday, spokesperso...",0
3,3,Vaccines Contain Nanobots That Track Your Loca...,RedPillReporter,"Wake up, sheeple! Hidden documents confirm tha...",1
4,4,Department of Labor Releases Monthly Jobs Report,Michael Torres,WASHINGTON — Officials confirmed Tuesday that ...,0


In [6]:
df.shape

(6000, 5)

In [7]:
df.isnull().sum()

,0
id,0
title,0
author,0
text,0
label,0


In [8]:
###Drop Nan Values
df=df.dropna()


In [9]:
df.head()

,id,title,author,text,label
0,0,Congress Approves Annual Defense Budget,Daniel Martinez,A new study published in the peer-reviewed jou...,0
1,1,Senate Passes Bipartisan Infrastructure Package,Paul Anderson,The annual report released Thursday demonstrat...,0
2,2,Study Finds Mediterranean Diet Linked to Longe...,Steven Robinson,"In a press conference held Monday, spokesperso...",0
3,3,Vaccines Contain Nanobots That Track Your Loca...,RedPillReporter,"Wake up, sheeple! Hidden documents confirm tha...",1
4,4,Department of Labor Releases Monthly Jobs Report,Michael Torres,WASHINGTON — Officials confirmed Tuesday that ...,0


In [10]:
## Get the Independent Features

X=df.drop('label',axis=1)

In [11]:
## Get the Dependent features
y=df['label']

In [12]:
X.shape

(6000, 4)

In [13]:
y.shape

(6000,)

In [14]:
import tensorflow as tf

In [15]:
tf.__version__

'2.19.0'

In [16]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [17]:
### Vocabulary size
voc_size=5000

### Onehot Representation

In [18]:
messages=X.copy()

In [19]:
messages['title'][1]

'Senate Passes Bipartisan Infrastructure Package'

In [20]:
messages

,id,title,author,text
0,0,Congress Approves Annual Defense Budget,Daniel Martinez,A new study published in the peer-reviewed jou...
1,1,Senate Passes Bipartisan Infrastructure Package,Paul Anderson,The annual report released Thursday demonstrat...
2,2,Study Finds Mediterranean Diet Linked to Longe...,Steven Robinson,"In a press conference held Monday, spokesperso..."
3,3,Vaccines Contain Nanobots That Track Your Loca...,RedPillReporter,"Wake up, sheeple! Hidden documents confirm tha..."
4,4,Department of Labor Releases Monthly Jobs Report,Michael Torres,WASHINGTON — Officials confirmed Tuesday that ...
...,...,...,...,...
5995,5995,Big Pharma Paying Doctors to Diagnose Healthy ...,UncensoredTruth,You won't believe what they are hiding! Source...
5996,5996,Big Pharma Paying Doctors to Diagnose Healthy ...,RealNewsNow,You won't believe what they are hiding! Source...
5997,5997,Hollywood Stars Running Satanic Cult Exposed b...,InsiderInformer,The mainstream media is REFUSING to cover the ...
5998,5998,Scientists CONFIRM: 5G Towers Spread New Virus,TruthSeeker99,"Wake up, sheeple! Hidden documents confirm tha..."


In [21]:
#messages.reset_index(inplace=True)

In [22]:
messages

,id,title,author,text
0,0,Congress Approves Annual Defense Budget,Daniel Martinez,A new study published in the peer-reviewed jou...
1,1,Senate Passes Bipartisan Infrastructure Package,Paul Anderson,The annual report released Thursday demonstrat...
2,2,Study Finds Mediterranean Diet Linked to Longe...,Steven Robinson,"In a press conference held Monday, spokesperso..."
3,3,Vaccines Contain Nanobots That Track Your Loca...,RedPillReporter,"Wake up, sheeple! Hidden documents confirm tha..."
4,4,Department of Labor Releases Monthly Jobs Report,Michael Torres,WASHINGTON — Officials confirmed Tuesday that ...
...,...,...,...,...
5995,5995,Big Pharma Paying Doctors to Diagnose Healthy ...,UncensoredTruth,You won't believe what they are hiding! Source...
5996,5996,Big Pharma Paying Doctors to Diagnose Healthy ...,RealNewsNow,You won't believe what they are hiding! Source...
5997,5997,Hollywood Stars Running Satanic Cult Exposed b...,InsiderInformer,The mainstream media is REFUSING to cover the ...
5998,5998,Scientists CONFIRM: 5G Towers Spread New Virus,TruthSeeker99,"Wake up, sheeple! Hidden documents confirm tha..."


In [23]:
import nltk
import re
from nltk.corpus import stopwords

In [24]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [25]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer ##stemming purpose
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [26]:
corpus

['congress approv annual defens budget',
 'senat pass bipartisan infrastructur packag',
 'studi find mediterranean diet link longer life',
 'vaccin contain nanobot track locat',
 'depart labor releas monthli job report',
 'health offici urg flu vaccin winter',
 'leak document reveal moon land film hollywood',
 'report declin global malaria case',
 'presid sign new econom relief bill law',
 'citi council vote new public transit plan',
 'nasa success launch new mar explor rover',
 'govern releas annual budget propos',
 'scientist confirm discoveri new exoplanet',
 'scientist confirm discoveri new exoplanet',
 'scientist secretli replac sun giant lamp decad ago',
 'exclus presid secretli arrest bodi doubl run countri',
 'new smartphon app steal soul pastor warn',
 'new smartphon app steal soul pastor warn',
 'local water suppli lace fluorid dumb popul',
 'citi council vote new public transit plan',
 'local water suppli lace fluorid dumb popul',
 'local govern approv commun hous project',


In [27]:
corpus[1]

'senat pass bipartisan infrastructur packag'

In [28]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[3426, 4469, 4823, 4171, 1811],
 [2005, 3248, 3496, 2464, 3926],
 [3421, 3562, 217, 3466, 932, 2506, 4698],
 [4003, 798, 1647, 1968, 3571],
 [3200, 2678, 1776, 4658, 1882, 2659],
 [2199, 959, 3241, 3510, 4003, 721],
 [3573, 4511, 4549, 3066, 2031, 2954, 4559],
 [2659, 2585, 4900, 66, 4057],
 [1408, 3758, 2706, 4336, 4062, 4413, 4435],
 [1222, 1745, 2150, 2706, 1828, 3054, 2045],
 [1029, 4276, 4718, 2706, 3300, 2064, 3255],
 [429, 1776, 4823, 1811, 2423],
 [4555, 141, 2215, 2706, 498],
 [4555, 141, 2215, 2706, 498],
 [4555, 2102, 1163, 4590, 2121, 1714, 1198, 1772],
 [458, 1408, 2102, 961, 3999, 4806, 663, 3556],
 [2706, 132, 2029, 735, 732, 540, 511],
 [2706, 132, 2029, 735, 732, 540, 511],
 [1843, 3812, 4613, 2712, 4709, 2052, 3086],
 [1222, 1745, 2150, 2706, 1828, 3054, 2045],
 [1843, 3812, 4613, 2712, 4709, 2052, 3086],
 [1843, 429, 4469, 2015, 4040, 2653],
 [2783, 4845, 2242, 1263, 2727, 3417, 2397],
 [4555, 141, 2215, 2706, 498],
 [3150, 1377, 2943, 762, 3667, 4057],
 [1408, 3758

In [29]:
corpus[1]

'senat pass bipartisan infrastructur packag'

In [30]:
onehot_repr[1]

[2005, 3248, 3496, 2464, 3926]

### Embedding Representation

In [31]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='post',maxlen=sent_length)
print(embedded_docs)

[[3426 4469 4823 ...    0    0    0]
 [2005 3248 3496 ...    0    0    0]
 [3421 3562  217 ...    0    0    0]
 ...
 [4559 2841  663 ...    0    0    0]
 [4555  141 3045 ...    0    0    0]
 [4041 4430 1745 ...    0    0    0]]


In [32]:
embedded_docs[1]

array([2005, 3248, 3496, 2464, 3926,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32)

In [33]:
embedded_docs[0]

array([3426, 4469, 4823, 4171, 1811,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32)

In [34]:
## Creating model
embedding_vector_features=40 ##features representation
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [35]:
len(embedded_docs),y.shape

(6000, (6000,))

In [36]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [37]:
X_final.shape,y_final.shape

((6000, 20), (6000,))

In [38]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

### Model Training

In [39]:
### Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.8818 - loss: 0.2323 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 6.5619e-04 - val_accuracy: 1.0000 - val_loss: 4.1905e-04
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 3.2501e-04 - val_accuracy: 1.0000 - val_loss: 2.5171e-04
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 2.0877e-04 - val_accuracy: 1.0000 - val_loss: 1.7274e-04
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 1.4900e-04 - val_accuracy: 1.0000 - val_loss: 1.2817e-04
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 1.1340e-04 - val_accuracy: 1.0000 - val_loss: 1.0005e-04
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 - loss: 9.0083e-05 - val_accuracy: 1.0000 - val_loss: 8.0883e-05
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 1.0000 

### Adding Dropout

In [40]:
from tensorflow.keras.layers import Dropout
## Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(Dropout(0.3))
model.add(LSTM(100))
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

### Performance Metrics And Accuracy

In [41]:
y_pred=model.predict(X_test)

62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [42]:
y_pred=np.where(y_pred > 0.6, 1,0) ##AUC ROC Curve

In [43]:
from sklearn.metrics import confusion_matrix

In [44]:
confusion_matrix(y_test,y_pred)

array([[ 976,    0],
       [1004,    0]])

In [45]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.49292929292929294

In [46]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.49      1.00      0.66       976
           1       0.00      0.00      0.00      1004

    accuracy                           0.49      1980
   macro avg       0.25      0.50      0.33      1980
weighted avg       0.24      0.49      0.33      1980



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
